# Projet final : les récoltes agricoles

**Équipe :** ekip001, DALEUS Roodlyson, MICHEL Marx Darwin, OLESTIL Roubens.

Une direction départementale de l'agriculture (fictive) vous confie trois ans de récoltes, de 2023 à 2025, par saison (printemps, automne), département et culture, avec la pluie tombée pendant chaque saison. Le ministère veut savoir quelles cultures pèsent le plus, où les rendements sont les meilleurs, et si la pluie explique les mauvaises années.

Ce notebook montre **votre démarche** : chaque étape dit ce que vous cherchez, le code, le résultat et ce que vous en concluez. Les graphiques doivent être visibles : avant de pousser, **Restart**, puis **Run All**.

Le tableau de bord (`app.py`) reprend ensuite vos résultats pour le décideur.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 100,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

brut = pd.read_csv("donnees/recoltes_brut.csv")
cultures = pd.read_csv("donnees/cultures.csv")
pluies = pd.read_csv("donnees/pluies.csv")
departements = pd.read_csv("donnees/departements.csv")

## 1. Diagnostic

Forme, types, valeurs manquantes, lignes en double, et toutes les écritures d'une même valeur dans les colonnes de texte. Notez chaque problème trouvé : vous le corrigerez à l'étape 2.

In [ ]:
# Forme et types : surface_ha et production_t devraient être des nombres
display(brut.shape)
display(brut.dtypes)

# Valeurs manquantes et lignes en double
display(brut.isna().sum())
display(f"{brut.duplicated().sum()} lignes en double (copies exactes d'une autre ligne)")

# Toutes les écritures d'une même valeur dans les colonnes de texte
for col in ["saison", "departement", "culture"]:
    display(f"{col} : {brut[col].nunique()} écritures différentes", sorted(brut[col].unique()))

# Nombres écrits comme du texte : on cherche ce qui n'est pas une suite de chiffres
surface_texte = brut[~brut["surface_ha"].str.fullmatch(r"\d+")]
production_texte = brut[brut["production_t"].notna() & ~brut["production_t"].str.fullmatch(r"\d+", na=False)]
display(surface_texte["surface_ha"].head(6).tolist(), production_texte["production_t"].head(6).tolist())

# Problèmes trouvés, à corriger à l'étape 2
pd.DataFrame(
    [
        ("Casse et espaces parasites dans departement", f"{brut['departement'].nunique()} écritures pour {departements['departement'].nunique()} départements"),
        ("Casse et espaces parasites dans culture", f"{brut['culture'].nunique()} écritures pour {cultures['culture'].nunique()} cultures"),
        ("Surfaces écrites en texte (unité « ha » ou espace des milliers)", f"{len(surface_texte)} lignes"),
        ("Productions écrites en texte (unité « t »)", f"{len(production_texte)} lignes"),
        ("Productions manquantes", f"{brut['production_t'].isna().sum()} lignes"),
        ("Lignes en double", f"{brut.duplicated().sum()} lignes"),
    ],
    columns=["problème", "constat"],
)

## 2. Nettoyage

Une copie `propre`, puis une cellule par problème, en comptant les lignes avant et après chaque suppression. Dites ce que vous supprimez et pourquoi.

**Contrôle :** à la fin, `propre.shape` vaut `(376, 6)`, sans valeur manquante ni doublon, avec 2 valeurs dans `saison`, 10 valeurs dans `departement`, 7 valeurs dans `culture`.

In [ ]:
# Une copie : brut reste intact pour pouvoir comparer
propre = brut.copy()

# --- Problème 1 : casse et espaces parasites (saison, departement, culture) ---
# On enlève les espaces et on passe en minuscules, puis on remet les noms officiels
# de departements.csv et cultures.csv. Aucune ligne n'est supprimée : on regroupe les écritures.
for col in ["saison", "departement", "culture"]:
    propre[col] = propre[col].str.strip().str.lower()
propre["departement"] = propre["departement"].map({d.lower(): d for d in departements["departement"]})
propre["culture"] = propre["culture"].map({c.lower(): c for c in cultures["culture"]})
propre["saison"] = propre["saison"].str.capitalize()
assert propre[["saison", "departement", "culture"]].notna().all().all()  # aucun nom inconnu
display({col: propre[col].nunique() for col in ["saison", "departement", "culture"]})

# --- Problème 2 : nombres écrits en texte ---
# On garde uniquement les chiffres (\D = tout sauf un chiffre : retire " ha", " t", espaces des milliers).
# Une chaîne vide devient une valeur manquante, traitée au problème 4.
for col in ["surface_ha", "production_t"]:
    chiffres = propre[col].astype("string").str.replace(r"\D", "", regex=True)
    propre[col] = pd.to_numeric(chiffres.replace("", pd.NA))
display(propre[["surface_ha", "production_t"]].dtypes)

# --- Problème 3 : lignes en double ---
# Fait APRÈS la normalisation : " Ouest" et "Ouest" étaient deux valeurs différentes avant.
# On supprime les copies exactes : sinon la même récolte serait comptée deux fois dans les totaux.
avant = len(propre)
propre = propre.drop_duplicates()
display(f"Doublons : {avant} → {len(propre)} lignes ({avant - len(propre)} copies exactes supprimées, la même récolte était comptée deux fois)")

# --- Problème 4 : productions manquantes ---
# On supprime ces lignes : sans production, on ne peut calculer ni rendement ni valeur,
# et inventer un chiffre fausserait les totaux.
avant = len(propre)
propre = propre.dropna().reset_index(drop=True)
propre["surface_ha"] = propre["surface_ha"].astype(int)
propre["production_t"] = propre["production_t"].astype(int)
display(f"Productions manquantes : {avant} → {len(propre)} lignes ({avant - len(propre)} lignes sans production supprimées)")

# --- Contrôle final ---
controle = {
    "forme": propre.shape,
    "valeurs manquantes": int(propre.isna().sum().sum()),
    "doublons": int(propre.duplicated().sum()),
    "saison": propre["saison"].nunique(),
    "departement": propre["departement"].nunique(),
    "culture": propre["culture"].nunique(),
}
assert controle["forme"] == (376, 6)
controle

## 3. Jointures

Joignez `propre` avec `cultures.csv`, `pluies.csv`, et ajoutez les colonnes calculées dont vous aurez besoin. Comptez les lignes avant et après chaque `merge`.

**Contrôle :** 376 lignes après les jointures, autant qu'avant.

In [ ]:
# Une copie : brut reste intact pour pouvoir comparer
propre = brut.copy()

# --- Problème 1 : casse et espaces parasites (saison, departement, culture) ---
# On enlève les espaces et on passe en minuscules, puis on remet les noms officiels
# de departements.csv et cultures.csv. Aucune ligne n'est supprimée : on regroupe les écritures.
for col in ["saison", "departement", "culture"]:
    propre[col] = propre[col].str.strip().str.lower()
propre["departement"] = propre["departement"].map({d.lower(): d for d in departements["departement"]})
propre["culture"] = propre["culture"].map({c.lower(): c for c in cultures["culture"]})
propre["saison"] = propre["saison"].str.capitalize()
assert propre[["saison", "departement", "culture"]].notna().all().all()  # aucun nom inconnu
display({col: propre[col].nunique() for col in ["saison", "departement", "culture"]})

# --- Problème 2 : nombres écrits en texte ---
# On garde uniquement les chiffres (\D = tout sauf un chiffre : retire " ha", " t", espaces des milliers).
# Une chaîne vide devient une valeur manquante, traitée au problème 4.
for col in ["surface_ha", "production_t"]:
    chiffres = propre[col].astype("string").str.replace(r"\D", "", regex=True)
    propre[col] = pd.to_numeric(chiffres.replace("", pd.NA))
display(propre[["surface_ha", "production_t"]].dtypes)

# --- Problème 3 : lignes en double ---
# Fait APRÈS la normalisation : " Ouest" et "Ouest" étaient deux valeurs différentes avant.
# On supprime les copies exactes : sinon la même récolte serait comptée deux fois dans les totaux.
avant = len(propre)
propre = propre.drop_duplicates()
display(f"Doublons : {avant} → {len(propre)} lignes ({avant - len(propre)} copies exactes supprimées, la même récolte était comptée deux fois)")

# --- Problème 4 : productions manquantes ---
# On supprime ces lignes : sans production, on ne peut calculer ni rendement ni valeur,
# et inventer un chiffre fausserait les totaux.
avant = len(propre)
propre = propre.dropna().reset_index(drop=True)
propre["surface_ha"] = propre["surface_ha"].astype(int)
propre["production_t"] = propre["production_t"].astype(int)
display(f"Productions manquantes : {avant} → {len(propre)} lignes ({avant - len(propre)} lignes sans production supprimées)")

# --- Contrôle final ---
controle = {
    "forme": propre.shape,
    "valeurs manquantes": int(propre.isna().sum().sum()),
    "doublons": int(propre.duplicated().sum()),
    "saison": propre["saison"].nunique(),
    "departement": propre["departement"].nunique(),
    "culture": propre["culture"].nunique(),
}
assert controle["forme"] == (376, 6)
controle

Enregistrez le tableau complet : c'est le fichier que lit le tableau de bord.

```python
complet.to_csv("donnees/recoltes_propre.csv", index=False)
```

## 4. Quelles cultures pèsent le plus, en tonnes puis en valeur (gourdes) ? Le classement change-t-il ?

**Ce qu'on en conclut :**

## 5. Où les rendements (tonnes par hectare) sont-ils les meilleurs, culture par culture ?

**Ce qu'on en conclut :**

## 6. La pluie explique-t-elle les rendements ? Pour quelles cultures ? Montrez-le avec des nuages de points.

**Ce qu'on en conclut :**

## 7. Quelles cultures progressent ou reculent entre 2023 et 2025 ?

**Ce qu'on en conclut :**

## 8. Rapportez la production à la population de chaque département : quels départements produisent le plus par habitant ?

**Ce qu'on en conclut :**

## 9. Conclusion

Cinq phrases au plus : ce qu'il faut retenir, ce que les données ne permettent pas de dire, ce qu'il faudrait mesurer ensuite.